In [4]:
!pip install nest_asyncio

In [7]:
import nest_asyncio
import pandas as pd
import pysus

nest_asyncio.apply()

anos = list(range(2015, 2025))
lista_dfs = []

# Colunas fixas principais identificadas no seu print
colunas_fixas = [
    "NU_ANO",
    "DT_NOTIFIC",
    "DT_OCOR",
    "SG_UF_NOT",
    "ID_MUNICIP",
    "SG_UF_OCOR",
    "ID_MN_OCOR",
    "SG_UF",
    "ID_MN_RESI",
    "CS_ZONA",
    "LOCAL_OCOR",
    "CS_SEXO",
    "ANO_NASC",
    "NU_IDADE_N",
    "CS_RACA",
    "CS_ESCOL_N",
    "SIT_CONJUG",
    "ORIENT_SEX",
    "OUT_VEZES",
    "NUM_ENVOLV",
    "AUTOR_SEXO",
    "AUTOR_ALCO",
]

# Prefixos que pegam automaticamente deficiências, violências, vínculos (REL_),
# procedimentos de saúde (PROC_) e encaminhamentos à delegacia/rede (ENC_)
prefixos_automaticos = (
    "DEF_",
    "TRAN_",
    "VIOL_",
    "SEX_",
    "REL_",
    "PROC_",
    "ENC_",
)

for ano in anos:
    print(f"A processar o ano {ano}...")
    df_ano = pysus.ftp.sinan(disease="VIOL", year=ano, as_dataframe=True)

    # Padroniza nomes das colunas (remove espaços invisíveis e deixa em maiúsculas)
    df_ano.columns = df_ano.columns.astype(str).str.strip().str.upper()

    # Filtra Sexo Feminino ('F') e Violência Sexual ('1' na coluna VIOL_SEXU)
    filtro = (df_ano["CS_SEXO"].astype(str).str.strip() == "F") & (
        df_ano["VIOL_SEXU"].astype(str).str.strip() == "1"
    )
    df_filtrado = df_ano.loc[filtro].copy()

    # Seleciona as colunas fixas + todas que começam com os prefixos do projeto
    cols_selecionadas = [
        col
        for col in df_filtrado.columns
        if col in colunas_fixas or col.startswith(prefixos_automaticos)
    ]
    df_filtrado = df_filtrado[cols_selecionadas]

    print(
        f"  -> {len(df_filtrado)} registos filtrados em {ano} ({len(cols_selecionadas)} colunas)."
    )
    lista_dfs.append(df_filtrado)

# Junta todos os anos (2015 a 2024)
df_sinan_final = pd.concat(lista_dfs, ignore_index=True)

# Decodifica a idade do padrão DATASUS (ex: 425 -> 25 anos; < 400 -> 0 anos)
df_sinan_final["NU_IDADE_N"] = pd.to_numeric(
    df_sinan_final["NU_IDADE_N"], errors="coerce"
)
df_sinan_final["idade_anos"] = df_sinan_final["NU_IDADE_N"].apply(
    lambda x: (
        x - 400
        if pd.notna(x) and 400 <= x < 500
        else (0 if pd.notna(x) and x < 400 else None)
    )
)

# Salva o arquivo final pronto para a análise
df_sinan_final.to_parquet(
    "sinan_violencia_sexual_2015_2024.parquet", index=False
)
print(
    f"\nConcluído com sucesso! Total de registos (2015-2024): {len(df_sinan_final)}"
)
print("Colunas salvas no arquivo Parquet:", list(df_sinan_final.columns))

A processar o ano 2015...


VIOLBR15.parquet: 0.00B [00:00, ?B/s]/1 [00:00<?, ?file/s]


  -> 24393 registos filtrados em 2015 (93 colunas).
A processar o ano 2016...


VIOLBR16.parquet: 100%|██████████| 8.88M/8.88M [00:02<00:00, 3.94MB/s]


  -> 27354 registos filtrados em 2016 (93 colunas).
A processar o ano 2017...


...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  -> 69450 registos filtrados em 2023 (93 colunas).
A processar o ano 2024...


VIOLBR24.parquet: 100%|██████████| 27.5M/27.5M [00:03<00:00, 8.08MB/s]


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  -> 73638 registos filtrados em 2024 (93 colunas).

Concluído com sucesso! Total de registos (2015-2024): 433517
Colunas salvas no arquivo Parquet: ['DT_NOTIFIC', 'NU_ANO', 'SG_UF_NOT', 'ID_MUNICIP', 'DT_OCOR', 'ANO_NASC', 'NU_IDADE_N', 'CS_SEXO', 'CS_RACA', 'CS_ESCOL_N', 'SG_UF', 'ID_MN_RESI', 'SIT_CONJUG', 'DEF_TRANS', 'DEF_FISICA', 'DEF_MENTAL', 'DEF_VISUAL', 'DEF_AUDITI', 'TRAN_MENT', 'TRAN_COMP', 'DEF_OUT', 'DEF_ESPEC', 'SG_UF_OCOR', 'ID_MN_OCOR', 'LOCAL_OCOR', 'OUT_VEZES', 'VIOL_FISIC', 'VIOL_PSICO', 'VIOL_TORT', 'VIOL_SEXU', 'VIOL_TRAF', 'VIOL_FINAN', 'VIOL_NEGLI', 'VIOL_INFAN', 'VIOL_LEGAL', 'VIOL_OUTR', 'VIOL_ESPEC', 'SEX_ASSEDI', 'SEX_ESTUPR', 'SEX_PUDOR', 'SEX_PORNO', 'SEX_EXPLO', 'SEX_OUTRO', 'SEX_ESPEC', 'PROC_DST', 'PROC_HIV', 'PROC_HEPB', 'PROC_SANG', 'PROC_SEMEN', 'PROC_VAGIN', 'PROC_CONTR', 'PROC_ABORT', 'NUM_ENVOLV', 'REL_SEXUAL', 'REL_PAI', 'REL_MAE', 'REL_PAD', 'REL_CONJ', 'REL_EXCON', 'REL_NAMO', 'REL_EXNAM', 'REL_FILHO', 'REL_DESCO', 'REL_IRMAO', 'REL_CONHEC', 'R

In [6]:
# Mostra todas as colunas de df_ano (sem precisar baixar de novo!)
print("Total de colunas:", len(df_ano.columns))
print([col for col in df_ano.columns if "VIOL" in col.upper() or "SEX" in col.upper() or "ESTUP" in col.upper()])
print("Primeiras 30 colunas:", list(df_ano.columns[:30]))

Total de colunas: 159
['CS_SEXO', 'VIOL_FISIC', 'VIOL_PSICO', 'VIOL_TORT', 'VIOL_SEXU', 'VIOL_TRAF', 'VIOL_FINAN', 'VIOL_NEGLI', 'VIOL_INFAN', 'VIOL_LEGAL', 'VIOL_OUTR', 'VIOL_ESPEC', 'SEX_ASSEDI', 'SEX_ESTUPR', 'SEX_PUDOR', 'SEX_PORNO', 'SEX_EXPLO', 'SEX_OUTRO', 'SEX_ESPEC', 'REL_SEXUAL', 'AUTOR_SEXO', 'ORIENT_SEX', 'VIOL_MOTIV']
Primeiras 30 colunas: ['TP_NOT', 'ID_AGRAVO', 'DT_NOTIFIC', 'SEM_NOT', 'NU_ANO', 'SG_UF_NOT', 'ID_MUNICIP', 'DT_OCOR', 'SEM_PRI', 'ANO_NASC', 'NU_IDADE_N', 'CS_SEXO', 'CS_GESTANT', 'CS_RACA', 'CS_ESCOL_N', 'SG_UF', 'ID_MN_RESI', 'ID_PAIS', 'NDUPLIC', 'DT_INVEST', 'ID_OCUPA_N', 'SIT_CONJUG', 'DEF_TRANS', 'DEF_FISICA', 'DEF_MENTAL', 'DEF_VISUAL', 'DEF_AUDITI', 'TRAN_MENT', 'TRAN_COMP', 'DEF_OUT']
